In [ ]:
import os
WRI_PROJECT_ROOT = os.environ.get("WRI_PROJECT_ROOT", "/home/shares/wwri-wildfire")

# This code reads in shape files for the Yukon given to us by  Luc Bibeau, Manager, Prevention and Mitigation Unit, 
# Community Services | Wildland Fire Management.

# Yukon Fire Resources Preparation
# Reads the Yukon attack bases and emergency services shapefiles.
# Filters the emergency services to retain only records with fire services and flags them as municipal or volunteer 
# based on a predefined list of municipal fire departments.
# Saves the cleaned emergency services data as a new shapefile.


import geopandas as gpd
import pandas as pd
import os

# Set pandas to show all rows
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Read the shapefiles
attack_bases = gpd.read_file(os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'raw', 'fire_stations', '2023', 'yukon_fire_resources', 'raw', 'yukon_attack_bases', 'Attack Bases.shp'))
emergency_services = gpd.read_file(os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'raw', 'fire_stations', '2023', 'yukon_fire_resources', 'raw', 'yukon_emergancy_services', 'Emergency_Services_by_Community.shp'))

# Filter to only show places with fire services
emergency_services = emergency_services[emergency_services['FIRE'] == 'x']

# Create list of places with municipal fire departments
municipal_fire_depts = [
    'Whitehorse',
    'Watson Lake',
    'Dawson City',
    'Mayo',
    'Faro',
    'Haines Junction',
    'Teslin',
    'Carmacks'
]

# Add new columns
emergency_services['municipal_fire_dept'] = emergency_services['PLACE_NAME'].isin(municipal_fire_depts)
emergency_services['volunteer_fire_dept'] = (emergency_services['FIRE'] == 'x') & (~emergency_services['municipal_fire_dept'])

# Create output directory if it doesn't exist
output_dir = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'yukon_fire_resources')
os.makedirs(output_dir, exist_ok=True)

# Save the updated emergency services dataframe as a shapefile
output_path = os.path.join(output_dir, 'yukon_fire_departments.shp')
emergency_services.to_file(output_path)

print("Saved updated emergency services shapefile to:", output_path)

print("\nComplete Attack Bases Dataset:")
print("=============================")
print(attack_bases)
print("\nColumns:", attack_bases.columns.tolist())

print("\n\nEmergency Services Dataset (Fire Services Only):")
print("==============================================")
print(emergency_services)
print("\nColumns:", emergency_services.columns.tolist())

print("\n\nFire Services Summary:")
print("=====================")
print(emergency_services[['PLACE_NAME', 'FIRE', 'municipal_fire_dept', 'volunteer_fire_dept']].sort_values('PLACE_NAME'))